[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/E.%20Integrated%2C%20Resilient%2C%20%26%20Modern%20Supply%20Chains%20%28The%20Frontier%29/101.%20The%20Supply%20Chain%20Finance%20%26%20Working%20Capital%20Optimization%20Problem/P101-Tier-5_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---


# Problem 101, Tier 5: Integrated Digital Twin for Supply Chain Finance

## Problem Description

This notebook builds a small, object-oriented digital-twin simulation for supply chain finance. It represents supply chain entities, associates financial sensor readings with them, synchronizes virtual state, calculates simple forecasts, and tests what-if scenarios. The workflow is an illustrative simulation; it is not a real-time production system or a trained machine-learning application.

### Learning objectives

By the end of the notebook, you should be able to:

- Represent entities, sensor readings, predictive-model metadata, and scenarios as data structures.
- Trace how sensor readings update an entity's virtual state.
- Interpret simple trend-based forecasts and a scenario simulation.
- Calculate system indicators and visualize simulated results.
- Distinguish illustrative assumptions from measurements or validated financial forecasts.

### Inputs and outputs

The inputs are six example entity records, randomly generated sensor values, four scenario definitions, and explicit assumptions in the simulation. The outputs are synchronized virtual states, forecasts, scenario trajectories, summary indicators, plots, and illustrative cost and return calculations. Random values are generated without a fixed seed, so numerical outputs can vary between runs.

### Scope and modeling assumptions

The model is a teaching example, not an operational digital twin. It has no live data connections, optimization solver, trained forecasting model, validated risk model, or measured runtime benchmark. Sensor readings are generated once during setup; synchronization applies them to virtual state but does not ingest a live stream. Sensor confidence is stored as metadata but is not used to weight or filter readings. When several sensors of a supported type are attached to one entity, the last registered reading of that type supplies the corresponding virtual-state field. The trend forecasts and scenario dynamics are deliberately simplified proxies. A payment-term change is applied as a percentage effect on cash flow for each simulated day; a demand change is applied in the same way. Scenario noise is random, and each scenario is simulated independently. The ROI and maturity calculations are illustrative calculations based on stated assumptions, not evidence of realized benefits or deployment readiness.

## Digital Twin Data Structures

A digital twin pairs a representation of a physical entity with a virtual state that can be updated and analyzed. Here, a sensor is a record containing an entity identifier, a measurement, a unit, a timestamp, and a confidence value. A scenario is a set of parameter changes and descriptive metadata. These structures organize the example data; defining a predictive-model record does not train or run a machine-learning model.

## Digital Twin Operations

The simulation follows a fixed sequence: create entities and sensors, synchronize each entity, calculate simple forecasts, simulate scenarios, and summarize the resulting indicators. For working capital, the model uses

$$\text{Working capital} = \text{Inventory} + \text{Receivables} - \text{Payables}.$$

For a current value $x_0$ and daily trend $r$, the forecast proxy at horizon $h$ days is $x_h = x_0(1 + rh)$. This is a linear extrapolation, not a fitted statistical model. In scenario simulation, let $C_t$ and $W_t$ denote cash flow and working capital after simulated day $t$. If $p$ is the payment-term change, $d$ is the demand change, and $\epsilon_t$ and $\eta_t$ are random daily variations, the code applies

$$C_t = C_{t-1}(1 + 0.01p)(1 + 0.01d)(1 + \epsilon_t), \qquad W_t = W_{t-1}(1 + \eta_t),$$

$$K_t = 0.0001|W_t|.$$

Here, $K_t$ is the simulated daily carrying cost. The two parameter effects are applied sequentially to the current cash flow, as shown by the product of factors. This recurrence describes the simulation rule; it does not establish a causal financial relationship. The code implements these steps directly.

In [ ]:
from dataclasses import dataclass, field
from datetime import datetime, timedelta
from typing import Any, Dict, List, Optional, Tuple
import random

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from matplotlib.ticker import FuncFormatter

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

print("Packages imported successfully for the digital-twin simulation.")

## Digital Twin Data Structures

The data classes separate stored information from the procedures that operate on it. `physical_state` and `virtual_state` are dictionaries of named values; they can contain different financial fields depending on the entity. The system-level record stores these entities together with sensors, model metadata, and scenarios. A sensor's identifier must be unique across the system, while its `entity_id` specifies which entity owns the reading.

In [ ]:
@dataclass
class FinancialSensor:
    """A financial measurement associated with a supply chain entity."""

    sensor_id: str
    sensor_type: str
    entity_id: str
    timestamp: datetime
    value: float
    unit: str
    confidence: float
    metadata: Dict[str, Any] = field(default_factory=dict)


@dataclass
class DigitalTwinEntity:
    """Physical and virtual state records for one supply chain entity."""

    entity_id: str
    entity_type: str
    physical_state: Dict[str, Any]
    virtual_state: Dict[str, Any]
    sensors: List[FinancialSensor] = field(default_factory=list)
    last_sync: datetime = field(default_factory=datetime.now)
    sync_quality: float = 1.0


@dataclass
class PredictiveModel:
    """Metadata describing a predictive model; no model is trained here."""

    model_id: str
    model_type: str
    parameters: Dict[str, Any]
    accuracy: float
    last_trained: datetime
    forecast_horizon: int


@dataclass
class Scenario:
    """Parameters and descriptive metadata for a what-if simulation."""

    scenario_id: str
    scenario_name: str
    parameters: Dict[str, Any]
    probability: float
    time_horizon: int
    description: str


@dataclass
class DigitalTwinSystem:
    """Container for the records used by the digital-twin simulation."""

    system_id: str
    entities: Dict[str, DigitalTwinEntity] = field(default_factory=dict)
    sensors: Dict[str, FinancialSensor] = field(default_factory=dict)
    models: Dict[str, PredictiveModel] = field(default_factory=dict)
    scenarios: Dict[str, Scenario] = field(default_factory=dict)
    sync_frequency: int = 1
    uptime: float = 99.7
    total_state_variables: int = 0


print("Digital-twin data structures defined.")

## Digital Twin Implementation

`IntegratedDigitalTwin` owns the system records and provides the main operations. Synchronization updates the virtual state from the sensors attached to one entity. The simulation quality factor is drawn from the interval $[0.90, 1.00]$ and scales cash-flow and inventory readings; payment-term readings are copied directly. For an entity with synchronization factor $q$, the supported updates are therefore $\text{cash flow} \leftarrow qv$, $\text{inventory} \leftarrow qv$, and $\text{payment terms} \leftarrow v$, where $v$ is the selected sensor value. If multiple sensors of one supported type are attached to an entity, the last sensor in the stored list supplies that field. This simple rule makes the code easy to trace, but it is not a statistical aggregation method.

The forecast method applies the linear trend equation defined above. For cash flow and working capital, the trend rates are drawn independently from their respective intervals. The risk score is the arithmetic mean of synchronization quality and two trend-stability terms: $[q + (1-|r|) + (1-|r_{WC}|)]/3$. It is an illustrative indicator, not a calibrated probability of loss. The scenario method returns one value per simulated day. Its `days` argument controls the number of simulated steps, while the scenario record also stores a descriptive time horizon.

In [ ]:
class IntegratedDigitalTwin:
    """Manage entities and run the illustrative supply chain finance simulation."""

    def __init__(self, system_id: str):
        self.system = DigitalTwinSystem(system_id=system_id)
        self.simulation_time = datetime.now()
        self.time_step = timedelta(hours=1)

    def add_entity(self, entity: DigitalTwinEntity) -> None:
        """Register an entity and count its stored state fields."""
        if entity.entity_id in self.system.entities:
            raise ValueError(f"Entity {entity.entity_id} is already registered.")
        self.system.entities[entity.entity_id] = entity
        self.system.total_state_variables += (
            len(entity.physical_state) + len(entity.virtual_state)
        )

    def add_sensor(self, sensor: FinancialSensor) -> None:
        """Register a sensor and attach it to its entity."""
        if sensor.entity_id not in self.system.entities:
            raise ValueError(f"Unknown entity: {sensor.entity_id}")
        if sensor.sensor_id in self.system.sensors:
            raise ValueError(f"Sensor {sensor.sensor_id} is already registered.")
        self.system.sensors[sensor.sensor_id] = sensor
        self.system.entities[sensor.entity_id].sensors.append(sensor)

    @staticmethod
    def _working_capital(state: Dict[str, Any]) -> float:
        """Return stored working capital or derive it from its components."""
        if "working_capital" in state:
            return float(state["working_capital"])
        inventory = float(state.get("inventory_value", 0))
        receivables = float(state.get("receivables", 0))
        payables = float(state.get("payables", 0))
        return inventory + receivables - payables

    def synchronize_entity(self, entity_id: str) -> float:
        """Apply attached sensor values to an entity's virtual state."""
        if entity_id not in self.system.entities:
            return 0.0

        entity = self.system.entities[entity_id]
        sync_quality = 0.95 + random.uniform(-0.05, 0.05)

        for sensor in entity.sensors:
            if sensor.sensor_type == "cash_flow":
                entity.virtual_state["cash_flow"] = sensor.value * sync_quality
            elif sensor.sensor_type == "inventory":
                entity.virtual_state["inventory_value"] = (
                    sensor.value * sync_quality
                )
            elif sensor.sensor_type == "payment":
                entity.virtual_state["payment_terms"] = sensor.value

        entity.sync_quality = sync_quality
        entity.last_sync = self.simulation_time
        return sync_quality

    def run_predictive_analytics(self, entity_id: str) -> Dict[str, float]:
        """Calculate simple trend-based forecasts and an illustrative risk score."""
        if entity_id not in self.system.entities:
            return {}

        entity = self.system.entities[entity_id]
        current_cash = float(entity.virtual_state.get("cash_flow", 0))
        trend = random.uniform(-0.02, 0.03)
        predictions = {
            "cash_flow_7d": current_cash * (1 + trend * 7),
            "cash_flow_30d": current_cash * (1 + trend * 30),
        }

        current_wc = self._working_capital(entity.virtual_state)
        wc_trend = random.uniform(-0.01, 0.02)
        predictions["working_capital_7d"] = current_wc * (1 + wc_trend * 7)
        predictions["working_capital_30d"] = current_wc * (1 + wc_trend * 30)

        risk_factors = [
            entity.sync_quality,
            1.0 - abs(trend),
            1.0 - abs(wc_trend),
        ]
        predictions["risk_score"] = float(np.mean(risk_factors))
        return predictions

    def simulate_scenario(
        self, scenario_id: str, entity_id: str, days: int = 30
    ) -> Dict[str, List[float]]:
        """Simulate daily cash flow, working capital, and carrying cost."""
        if days < 0:
            raise ValueError("days must be non-negative.")
        if (
            scenario_id not in self.system.scenarios
            or entity_id not in self.system.entities
        ):
            return {}

        scenario = self.system.scenarios[scenario_id]
        entity = self.system.entities[entity_id]
        results = {"cash_flow": [], "working_capital": [], "cost": []}
        cash_flow = float(entity.virtual_state.get("cash_flow", 100000))
        working_capital = self._working_capital(entity.virtual_state)

        for _ in range(days):
            payment_change = float(
                scenario.parameters.get("payment_term_change", 0)
            )
            demand_change = float(scenario.parameters.get("demand_change", 0))
            cash_flow += payment_change * 0.01 * cash_flow
            cash_flow += demand_change * 0.01 * cash_flow
            cash_flow *= 1 + random.uniform(-0.02, 0.02)
            working_capital *= 1 + random.uniform(-0.01, 0.01)

            daily_cost = abs(working_capital) * 0.0001
            results["cash_flow"].append(cash_flow)
            results["working_capital"].append(working_capital)
            results["cost"].append(daily_cost)

        return results

    def get_system_kpi(self) -> Dict[str, float]:
        """Aggregate basic financial values and counts across entities."""
        if not self.system.entities:
            return {}

        entities = list(self.system.entities.values())
        total_cash_flow = sum(
            float(entity.virtual_state.get("cash_flow", 0)) for entity in entities
        )
        total_working_capital = sum(
            self._working_capital(entity.virtual_state) for entity in entities
        )
        return {
            "total_cash_flow": total_cash_flow,
            "total_working_capital": total_working_capital,
            "average_sync_quality": float(
                np.mean([entity.sync_quality for entity in entities])
            ),
            "total_entities": len(entities),
            "total_sensors": len(self.system.sensors),
            "state_variables": self.system.total_state_variables,
        }


print("Integrated digital-twin operations defined.")

## Digital Twin Setup

The setup creates six example entities, generates between 15 and 25 sensors of each of five types for every entity, and registers model and scenario metadata. Sensor identifiers include the full entity identifier so that every registered sensor has a unique system-wide key. The sensor total therefore depends on random draws; the notebook does not assume that every run has a particular sensor count. State-variable counts refer only to keys in the physical and virtual state dictionaries, not to every internal value in the system.

In [ ]:
dt_engine = IntegratedDigitalTwin("SCF-DT-001")

entities_config = [
    ("SUP-001", "supplier", {
        "inventory_value": 250000, "receivables": 180000, "payables": 120000,
        "cash_flow": 80000, "payment_terms": 45, "credit_score": 750,
    }),
    ("MAN-001", "manufacturer", {
        "inventory_value": 450000, "receivables": 320000, "payables": 280000,
        "cash_flow": 120000, "payment_terms": 60, "credit_score": 780,
    }),
    ("DIST-001", "distributor", {
        "inventory_value": 180000, "receivables": 150000, "payables": 100000,
        "cash_flow": 95000, "payment_terms": 30, "credit_score": 720,
    }),
    ("RET-001", "retailer", {
        "inventory_value": 120000, "receivables": 95000, "payables": 85000,
        "cash_flow": 110000, "payment_terms": 15, "credit_score": 700,
    }),
    ("FIN-001", "financial_institution", {
        "loan_portfolio": 2500000, "interest_income": 125000,
        "risk_exposure": 180000, "cash_flow": 200000,
        "payment_terms": 0, "credit_score": 850,
    }),
    ("LOG-001", "logistics_provider", {
        "fleet_value": 800000, "receivables": 200000, "payables": 150000,
        "cash_flow": 70000, "payment_terms": 30, "credit_score": 710,
    }),
]

for entity_id, entity_type, physical_state in entities_config:
    entity = DigitalTwinEntity(
        entity_id=entity_id,
        entity_type=entity_type,
        physical_state=physical_state.copy(),
        virtual_state=physical_state.copy(),
    )
    dt_engine.add_entity(entity)

sensor_types = ["payment", "inventory", "cash_flow", "market", "risk"]
for entity_id in dt_engine.system.entities:
    for sensor_type in sensor_types:
        sensors_per_type = random.randint(15, 25)
        for index in range(sensors_per_type):
            sensor_id = f"{sensor_type.upper()}-{entity_id}-{index + 1:03d}"

            if sensor_type == "payment":
                value, unit = random.uniform(15, 90), "days"
            elif sensor_type == "inventory":
                value, unit = random.uniform(50000, 500000), "USD"
            elif sensor_type == "cash_flow":
                value, unit = random.uniform(50000, 200000), "USD"
            elif sensor_type == "market":
                value, unit = random.uniform(0.5, 1.0), "score"
            else:
                value, unit = random.uniform(0.1, 0.9), "score"

            sensor = FinancialSensor(
                sensor_id=sensor_id,
                sensor_type=sensor_type,
                entity_id=entity_id,
                timestamp=datetime.now(),
                value=value,
                unit=unit,
                confidence=random.uniform(0.85, 0.99),
            )
            dt_engine.add_sensor(sensor)

models_config = [
    ("CF-MODEL-001", "cash_flow", {"algorithm": "LSTM", "window": 30}, 0.92),
    ("WD-MODEL-001", "working_capital", {"algorithm": "ARIMA", "window": 60}, 0.88),
    ("RK-MODEL-001", "risk", {"algorithm": "RandomForest", "window": 90}, 0.90),
    ("DM-MODEL-001", "demand", {"algorithm": "Prophet", "window": 120}, 0.85),
]
for model_id, model_type, parameters, accuracy in models_config:
    dt_engine.system.models[model_id] = PredictiveModel(
        model_id=model_id,
        model_type=model_type,
        parameters=parameters,
        accuracy=accuracy,
        last_trained=datetime.now() - timedelta(days=random.randint(1, 30)),
        forecast_horizon=30,
    )

scenarios_config = [
    ("SCEN-001", "Extended Payment Terms",
     {"payment_term_change": -15, "demand_change": 0}, 0.30, 30,
     "Extend supplier payment terms by 15 days"),
    ("SCEN-002", "Demand Surge",
     {"payment_term_change": 0, "demand_change": 20}, 0.20, 30,
     "20% increase in customer demand"),
    ("SCEN-003", "Market Disruption",
     {"payment_term_change": 10, "demand_change": -15}, 0.15, 30,
     "Supply chain disruption with tighter terms"),
    ("SCEN-004", "Optimization Initiative",
     {"payment_term_change": -5, "demand_change": 5}, 0.35, 30,
     "Working capital optimization program"),
]
for scenario_id, name, parameters, probability, horizon, description in scenarios_config:
    dt_engine.system.scenarios[scenario_id] = Scenario(
        scenario_id=scenario_id,
        scenario_name=name,
        parameters=parameters,
        probability=probability,
        time_horizon=horizon,
        description=description,
    )

assert len(dt_engine.system.entities) == len(entities_config)
assert len(dt_engine.system.sensors) == sum(
    len(entity.sensors) for entity in dt_engine.system.entities.values()
)
print("Digital-twin setup complete.")
print(f"Entities: {len(dt_engine.system.entities)}")
print(f"Sensors: {len(dt_engine.system.sensors)}")
print(f"Predictive-model records: {len(dt_engine.system.models)}")
print(f"Scenarios: {len(dt_engine.system.scenarios)}")
print(f"Stored state fields: {dt_engine.system.total_state_variables}")

## Single-Pass Digital Twin Operations

This section performs one synchronization pass, calculates forecasts for each entity, and aggregates system indicators. The operation is a single simulated pass rather than a continuously running real-time service. KPI working capital is derived consistently from inventory, receivables, and payables when an explicit `working_capital` field is absent. Synchronization quality describes the random scaling factor used in that pass; it is not a measured data-quality score.

In [ ]:
print("Starting one digital-twin simulation pass.\n")

sync_results = {}
for entity_id in dt_engine.system.entities:
    sync_results[entity_id] = dt_engine.synchronize_entity(entity_id)
    print(f"{entity_id}: synchronization quality = {sync_results[entity_id]:.3f}")

average_sync_quality = float(np.mean(list(sync_results.values())))
print(f"\nAverage synchronization quality: {average_sync_quality:.3f}")

predictions_summary = {}
print("\nTrend-based forecast proxies:")
for entity_id in dt_engine.system.entities:
    predictions = dt_engine.run_predictive_analytics(entity_id)
    predictions_summary[entity_id] = predictions
    print(f"\n{entity_id}:")
    print(f"  Cash flow, 7-day proxy: ${predictions['cash_flow_7d']:,.0f}")
    print(f"  Cash flow, 30-day proxy: ${predictions['cash_flow_30d']:,.0f}")
    print(f"  Working capital, 7-day proxy: ${predictions['working_capital_7d']:,.0f}")
    print(f"  Working capital, 30-day proxy: ${predictions['working_capital_30d']:,.0f}")
    print(f"  Illustrative risk score: {predictions['risk_score']:.3f}")

system_kpi = dt_engine.get_system_kpi()
print("\nSystem indicators:")
print(f"  Total cash flow: ${system_kpi['total_cash_flow']:,.0f}")
print(f"  Total working capital: ${system_kpi['total_working_capital']:,.0f}")
print(f"  Average synchronization quality: {system_kpi['average_sync_quality']:.3f}")
print(f"  Entities: {system_kpi['total_entities']}")
print(f"  Sensors: {system_kpi['total_sensors']}")
print(f"  Stored state fields: {system_kpi['state_variables']}")

total_entities = len(dt_engine.system.entities)
total_sensors = len(dt_engine.system.sensors)
avg_sensors_per_entity = total_sensors / total_entities
state_fields_per_entity = dt_engine.system.total_state_variables / total_entities
print("\nDescriptive system counts:")
print(f"  Sensors per entity: {avg_sensors_per_entity:.1f}")
print(f"  Stored state fields per entity: {state_fields_per_entity:.1f}")
print(f"  Configured uptime assumption: {dt_engine.system.uptime}%")
print(f"  Configured synchronization interval: {dt_engine.system.sync_frequency} second(s)")

## Scenario Analysis and What-If Testing

Each scenario is simulated for 30 daily steps on the manufacturer entity. The reported percentage changes compare the final simulated values with that entity's state at the start of scenario analysis. The working-capital baseline is computed from its components using the same rule as the KPI calculation. Scenario probabilities are supplied metadata; a probability-weighted expected outcome is not calculated here. Because each trajectory contains independent random variation, a single run is one illustrative realization rather than a robust estimate of scenario impact.

For each day, the simulation starts from the previous day's cash flow and working capital, applies the scenario parameter effects and random variation, and records the resulting values. The resulting cost list contains one carrying-cost estimate per simulated day; summing it gives the total modeled carrying cost over the simulated path. These calculations are conditional on the stated rules and do not represent a calibrated financial model.

In [ ]:
test_entity = "MAN-001"
scenario_results = {}
baseline_state = dt_engine.system.entities[test_entity].virtual_state
baseline_cash = float(baseline_state.get("cash_flow", 0))
baseline_wc = dt_engine._working_capital(baseline_state)

print("Scenario analysis for the manufacturer entity.\n")
for scenario_id, scenario in dt_engine.system.scenarios.items():
    results = dt_engine.simulate_scenario(scenario_id, test_entity, days=30)
    scenario_results[scenario_id] = results
    final_cash = results["cash_flow"][-1]
    final_wc = results["working_capital"][-1]
    total_cost = sum(results["cost"])
    cash_change = (final_cash - baseline_cash) / baseline_cash * 100
    wc_change = (final_wc - baseline_wc) / baseline_wc * 100 if baseline_wc else 0.0

    print(f"{scenario.scenario_name}")
    print(f"  Description: {scenario.description}")
    print(f"  Supplied scenario probability: {scenario.probability:.1%}")
    print(f"  Final simulated cash flow: ${final_cash:,.0f} ({cash_change:+.1f}%)")
    print(f"  Final simulated working capital: ${final_wc:,.0f} ({wc_change:+.1f}%)")
    print(f"  Sum of simulated daily carrying costs: ${total_cost:,.0f}\n")

print("Comparative scenario summary:")
print(f"{'Scenario':<28} | {'Cash change':>12} | {'WC change':>10} | {'Cost':>12} | {'Probability':>11}")
print("-" * 86)
for scenario_id, scenario in dt_engine.system.scenarios.items():
    results = scenario_results[scenario_id]
    final_cash = results["cash_flow"][-1]
    final_wc = results["working_capital"][-1]
    total_cost = sum(results["cost"])
    cash_change = (final_cash - baseline_cash) / baseline_cash * 100
    wc_change = (final_wc - baseline_wc) / baseline_wc * 100 if baseline_wc else 0.0
    print(
        f"{scenario.scenario_name[:28]:<28} | {cash_change:+11.1f}% | "
        f"{wc_change:+9.1f}% | ${total_cost:10,.0f} | {scenario.probability:10.1%}"
    )

best_scenario_id = max(
    scenario_results,
    key=lambda scenario_id: scenario_results[scenario_id]["cash_flow"][-1],
)
worst_scenario_id = min(
    scenario_results,
    key=lambda scenario_id: scenario_results[scenario_id]["cash_flow"][-1],
)
print("\nHighest final simulated cash flow:")
print(
    f"  {dt_engine.system.scenarios[best_scenario_id].scenario_name}: "
    f"${scenario_results[best_scenario_id]['cash_flow'][-1]:,.0f}"
)
print("Lowest final simulated cash flow:")
print(
    f"  {dt_engine.system.scenarios[worst_scenario_id].scenario_name}: "
    f"${scenario_results[worst_scenario_id]['cash_flow'][-1]:,.0f}"
)

## Digital Twin Visualization Dashboard

The dashboard displays synchronization quality, current and forecast cash-flow proxies, scenario changes, and system counts. Scenario bars use the same baseline and final simulated values as the printed comparison. The performance panel labels each metric's scaling explicitly; its bars are not measurements on a shared physical scale. The plotted forecasts and scenario changes inherit the randomness of the run, so the figure should be read as a view of this simulation realization rather than as a stable estimate.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle(
    "Illustrative Digital-Twin Simulation: Supply Chain Finance",
    fontsize=16,
    fontweight="bold",
)

entity_ids = list(sync_results)
sync_qualities = [sync_results[entity_id] for entity_id in entity_ids]
entity_types = [
    dt_engine.system.entities[entity_id].entity_type for entity_id in entity_ids
]
colors = ["#FF6B6B", "#4ECDC4", "#45B7D1", "#96CEB4", "#FFEAA7", "#DDA0DD"]
entity_colors = [colors[index % len(colors)] for index in range(len(entity_ids))]
bars = axes[0, 0].bar(entity_ids, sync_qualities, color=entity_colors, alpha=0.8)
axes[0, 0].set_title("Entity Synchronization Quality")
axes[0, 0].set_ylabel("Quality score (unitless)")
axes[0, 0].set_ylim(0.8, 1.02)
axes[0, 0].tick_params(axis="x", rotation=45)
for bar, entity_type in zip(bars, entity_types):
    axes[0, 0].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.003,
        entity_type,
        ha="center",
        va="bottom",
        fontsize=8,
    )

current_cash = [
    dt_engine.system.entities[entity_id].virtual_state.get("cash_flow", 0)
    for entity_id in entity_ids
]
cash_flow_7d = [predictions_summary[entity_id]["cash_flow_7d"] for entity_id in entity_ids]
cash_flow_30d = [predictions_summary[entity_id]["cash_flow_30d"] for entity_id in entity_ids]
x = np.arange(len(entity_ids))
width = 0.25
axes[0, 1].bar(x - width, current_cash, width, label="Current", color="lightblue")
axes[0, 1].bar(x, cash_flow_7d, width, label="7-day proxy", color="blue")
axes[0, 1].bar(x + width, cash_flow_30d, width, label="30-day proxy", color="darkblue")
axes[0, 1].set_title("Cash-Flow Trend Proxies")
axes[0, 1].set_ylabel("Cash flow (USD)")
axes[0, 1].set_xticks(x)
axes[0, 1].set_xticklabels(entity_ids, rotation=45)
axes[0, 1].yaxis.set_major_formatter(
    FuncFormatter(lambda value, position: f"${value / 1000:.0f}K")
)
axes[0, 1].legend()

scenario_names = []
cash_changes = []
wc_changes = []
for scenario_id, scenario in dt_engine.system.scenarios.items():
    results = scenario_results[scenario_id]
    scenario_names.append(scenario.scenario_name)
    cash_changes.append(
        (results["cash_flow"][-1] - baseline_cash) / baseline_cash * 100
    )
    wc_changes.append(
        (results["working_capital"][-1] - baseline_wc) / baseline_wc * 100
        if baseline_wc else 0.0
    )
x_scenarios = np.arange(len(scenario_names))
scenario_width = 0.35
axes[1, 0].bar(
    x_scenarios - scenario_width / 2,
    cash_changes,
    scenario_width,
    label="Cash-flow change",
    color="green",
    alpha=0.7,
)
axes[1, 0].bar(
    x_scenarios + scenario_width / 2,
    wc_changes,
    scenario_width,
    label="Working-capital change",
    color="orange",
    alpha=0.7,
)
axes[1, 0].set_title("Scenario Changes After 30 Simulated Days")
axes[1, 0].set_ylabel("Change from baseline (%)")
axes[1, 0].set_xticks(x_scenarios)
axes[1, 0].set_xticklabels(scenario_names, rotation=35, ha="right")
axes[1, 0].axhline(y=0, color="black", linewidth=0.8, alpha=0.5)
axes[1, 0].legend()

metrics_data = {
    "Entities": len(dt_engine.system.entities),
    "Sensors": len(dt_engine.system.sensors),
    "Stored state fields": dt_engine.system.total_state_variables,
    "Model records": len(dt_engine.system.models),
    "Scenarios": len(dt_engine.system.scenarios),
    "Mean sync quality (%)": system_kpi["average_sync_quality"] * 100,
}
scales = {
    "Entities": 20,
    "Sensors": 0.1,
    "Stored state fields": 0.1,
    "Model records": 20,
    "Scenarios": 20,
    "Mean sync quality (%)": 1,
}
metric_names = list(metrics_data)
normalized_values = [metrics_data[name] * scales[name] for name in metric_names]
metric_bars = axes[1, 1].bar(metric_names, normalized_values, color="purple", alpha=0.7)
axes[1, 1].set_title("System Counts on an Illustrative Scaled Axis")
axes[1, 1].set_ylabel("Scaled display value (not a common physical unit)")
axes[1, 1].tick_params(axis="x", rotation=35)
for bar, name in zip(metric_bars, metric_names):
    axes[1, 1].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{metrics_data[name]:,.1f}",
        ha="center",
        va="bottom",
        fontsize=8,
    )

plt.tight_layout()
plt.show()
print("Simulation dashboard created.")

## Advanced Digital Twin Analytics

This section examines sensor coverage and confidence, stored model metadata, simple processing-size estimates, an assumption-based ROI calculation, and a rule-based maturity rubric. These quantities describe the example and its assumptions. In particular, the model accuracy values are supplied metadata, the data-volume calculation assumes eight bytes per stored state field, and the ROI inputs are not measured benefits. Sensor coverage counts records, not unique measurement times; the average sensor value combines records of a given type and should be interpreted in that type's stated units.

In [ ]:
print("Advanced simulation analytics\n")

sensor_analysis = {}
for sensor_type in sensor_types:
    type_sensors = [
        sensor
        for sensor in dt_engine.system.sensors.values()
        if sensor.sensor_type == sensor_type
    ]
    if type_sensors:
        sensor_analysis[sensor_type] = {
            "count": len(type_sensors),
            "average_confidence": float(
                np.mean([sensor.confidence for sensor in type_sensors])
            ),
            "average_value": float(np.mean([sensor.value for sensor in type_sensors])),
            "sensors_per_entity": len(type_sensors) / len(dt_engine.system.entities),
        }

print("Sensor network summary:")
for sensor_type, analysis in sensor_analysis.items():
    print(
        f"  {sensor_type.capitalize():<12}: {analysis['count']:3d} records | "
        f"mean confidence: {analysis['average_confidence']:.3f} | "
        f"records per entity: {analysis['sensors_per_entity']:.1f}"
    )

print("\nPredictive-model metadata (accuracy values are supplied, not measured here):")
for model in dt_engine.system.models.values():
    days_since_training = (datetime.now() - model.last_trained).days
    print(
        f"  {model.model_type:<16}: metadata accuracy {model.accuracy:.1%} | "
        f"horizon {model.forecast_horizon} days | "
        f"last-trained field {days_since_training} days ago"
    )

total_sync_time_estimate = (
    len(dt_engine.system.entities) * dt_engine.system.sync_frequency
)
data_volume_estimate = dt_engine.system.total_state_variables * 8
processing_load_estimate = (
    data_volume_estimate / dt_engine.system.sync_frequency
    if dt_engine.system.sync_frequency > 0
    else float("inf")
)
print("\nIllustrative processing estimates:")
print(f"  Sequential sync-time estimate: {total_sync_time_estimate} seconds")
print(f"  State-field payload estimate: {data_volume_estimate / 1024:.1f} KB")
print(f"  Payload rate at configured interval: {processing_load_estimate:.1f} bytes/second")
print("  These estimates do not measure execution time, network traffic, or system capacity.")

print("\nAssumption-based ROI illustration:")
implementation_cost = 500000
annual_maintenance = 100000
sensor_cost = len(dt_engine.system.sensors) * 500
total_cost = implementation_cost + annual_maintenance + sensor_cost
working_capital_reduction = system_kpi["total_working_capital"] * 0.05
cash_flow_improvement = system_kpi["total_cash_flow"] * 0.03
risk_reduction = total_cost * 0.10
annual_benefits = (
    working_capital_reduction + cash_flow_improvement + risk_reduction
)
roi = (annual_benefits - annual_maintenance) / total_cost * 100
payback_period = total_cost / annual_benefits if annual_benefits > 0 else float("inf")
print(f"  Assumed implementation cost: ${implementation_cost:,.0f}")
print(f"  Assumed sensor cost: ${sensor_cost:,.0f}")
print(f"  Assumed annual maintenance: ${annual_maintenance:,.0f}")
print(f"  Combined cost basis: ${total_cost:,.0f}")
print(f"  Assumption-based annual benefit: ${annual_benefits:,.0f}")
print(f"  Illustrative ROI: {roi:.1f}%")
print(f"  Illustrative payback period: {payback_period:.1f} years")
print("  The calculation is sensitive to its assumptions and is not an investment appraisal.")

criteria = {
    "Synchronization quality": 20 if system_kpi["average_sync_quality"] > 0.95 else 10,
    "Sensor count": 20 if len(dt_engine.system.sensors) > 400 else 10,
    "Model metadata average": 20 if np.mean(
        [model.accuracy for model in dt_engine.system.models.values()]
    ) > 0.9 else 10,
    "Scenario count": 20 if len(dt_engine.system.scenarios) >= 4 else 10,
    "Stored state fields": 20 if dt_engine.system.total_state_variables > 5000 else 10,
}
maturity_score = sum(criteria.values())
print("\nIllustrative rule-based maturity rubric:")
for criterion, score in criteria.items():
    print(f"  {criterion:<28}: {score}/20")
print(f"  Rubric total: {maturity_score}/100")
if maturity_score >= 90:
    maturity_level = "Advanced example score"
elif maturity_score >= 70:
    maturity_level = "Mature example score"
elif maturity_score >= 50:
    maturity_level = "Developing example score"
else:
    maturity_level = "Basic example score"
print(f"  Rubric category: {maturity_level}")
print("  This rubric is illustrative and does not certify operational maturity.")

## Summary and Conclusions

This notebook demonstrates an object-oriented workflow for organizing supply chain finance data and running a small stochastic simulation. Its key steps are entity and sensor registration, one synchronization pass, trend-based forecast proxies, scenario simulation, KPI aggregation, visualization, and assumption-based analytics.

### Interpreting the results

- Sensor counts, synchronization values, forecasts, scenario trajectories, and sensor summaries are generated when the notebook runs. Because no random seed is set, they may differ across executions.
- The forecast equations are simple extrapolations, and the scenario trajectories include independent random variation. A single simulated path is not a prediction interval or a validated estimate of causal impact.
- The scenario probability fields are stored for context but are not used to calculate a probability-weighted expected outcome.
- The model records contain illustrative algorithm names and accuracy values; the notebook does not fit, evaluate, or call those algorithms.
- The processing, ROI, and maturity outputs are based on explicit simplifying assumptions. They should not be interpreted as measured system performance, realized financial return, or production readiness.

### Model boundaries

The example omits live sensor ingestion, data-quality filtering, time-series history, calibration, uncertainty analysis, and operational integration. The synchronization and scenario procedures are teaching mechanisms for tracing how data and assumptions affect virtual state. Extending the example for a real organization would require validated data, domain-specific financial relationships, and empirical testing; those activities are outside this notebook's scope.

### Final assessment

The notebook illustrates how a digital-twin-style data structure can bring entity state, sensor records, forecasts, and what-if simulations into one workflow. Its value here is educational: readers can inspect each assumption and follow how the code transforms inputs into outputs. The example does not establish that a production digital twin, a financial optimization model, or a deployable decision-support system has been built.